# Court Citation Concept Enrichment Smoke Test — 10 Rows

This notebook reads `court_considerations.csv`, enriches 10 court citation rows with **query-neutral legal descriptors**, and writes JSONL/CSV preview files.

Design choices:
- No synthetic user questions.
- No long summaries.
- Focus on specificity: concepts, original legal terms, topic/subtopic/micro-topic, statute/case anchors, legal test, fact-pattern tags.
- Selects substantive rows by default instead of the first 10 tiny/fragmentary rows.


In [ ]:
# Kaggle setup cell.
# Run this cell once, then restart the Kaggle session/kernel before running the rest.
# It removes FlashInfer because your Kaggle environment fails when FlashInfer JIT links libcuda.so.

!pip install -q -U "transformers>=4.45.0" accelerate safetensors pandas tqdm gptqmodel
!pip install -q -U "vllm>=0.6.0" || true
!pip uninstall -y flashinfer flashinfer-python || true

print("Setup done. Now restart the Kaggle session/kernel, then run from the next cell.")


In [ ]:
from pathlib import Path
import os
import re
import json
import time
import traceback
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm

@dataclass
class Config:
    # Kaggle competition path shown in your run log.
    input_csv: str = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv"
    fallback_input_csv: str = "court_considerations.csv"

    output_dir: str = "/kaggle/working"
    output_jsonl: str = "enriched_court_citations_10.jsonl"
    output_preview_csv: str = "enriched_court_citations_10_preview.csv"
    output_failures_jsonl: str = "enriched_court_citations_10_failures.jsonl"

    n_rows: int = 10

    # Use only substantive rows for the smoke test.
    min_text_chars: int = 600
    sample_random: bool = False
    random_seed: Optional[int] = 42

    # Input text budget per citation.
    # Keep this below max_model_len after chat-template overhead.
    text_chars: int = 3000

    # Model settings. Uses the local Kaggle model path, not Hugging Face Hub.
    model_name: str = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1"

    # vLLM-first, Transformers fallback.
    use_vllm_first: bool = True
    force_transformers: bool = False

    # Safe Kaggle T4 defaults.
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.62
    max_model_len: int = 3072
    max_num_seqs: int = 32
    enforce_eager: bool = True
    quantization: str = "awq"
    disable_custom_all_reduce: bool = True

    # Generation settings.
    batch_size: int = 4
    max_new_tokens: int = 384
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

    # Backend controls.
    # We actively avoid FlashInfer because this Kaggle image fails with:
    # /usr/bin/ld: cannot find -lcuda
    avoid_flashinfer: bool = True
    force_triton_attention: bool = True

cfg = Config()

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

# Do NOT set VLLM_ATTENTION_BACKEND here; your vLLM build reports it as unknown.
# The model-loading cell tries supported Python API options and falls back safely.

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(asdict(cfg))


In [ ]:
def resolve_input_path(cfg: Config) -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path("/kaggle/working") / cfg.fallback_input_csv,
        Path("/mnt/data") / cfg.fallback_input_csv,
        Path("/kaggle/working/court_consideration.csv"),
        Path("/mnt/data/court_consideration.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p

    if Path("/kaggle/input").exists():
        for pat in ["**/court_considerations.csv", "**/court_consideration.csv"]:
            found = sorted(Path("/kaggle/input").glob(pat))
            if found:
                return found[0]

    raise FileNotFoundError("Could not find court_considerations.csv. Update cfg.input_csv.")

input_path = resolve_input_path(cfg)
print("Using input:", input_path)

df = pd.read_csv(input_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)


In [ ]:
def pick_column(columns: List[str], preferred: List[str], contains_any: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        lc = c.lower()
        if any(token in lc for token in contains_any):
            return c
    return None

columns = list(df.columns)

citation_col = pick_column(
    columns,
    preferred=["citation", "cite", "court_citation", "authority_citation", "consideration_citation"],
    contains_any=["citation", "cite", "bge"],
)
text_col = pick_column(
    columns,
    preferred=["text", "consideration_text", "paragraph_text", "content", "raw_text", "body"],
    contains_any=["text", "content", "paragraph", "consideration", "body"],
)

if citation_col is None:
    raise ValueError("Could not infer citation column. Set citation_col manually.")
if text_col is None:
    raise ValueError("Could not infer text column. Set text_col manually.")

print("citation_col:", citation_col)
print("text_col:", text_col)

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid["_text_len"] = valid[text_col].str.strip().str.len()

# Use substantive rows for a meaningful smoke test.
valid = valid[valid["_text_len"] >= cfg.min_text_chars].copy()

if len(valid) < cfg.n_rows:
    raise ValueError(f"Only {len(valid)} rows have text length >= {cfg.min_text_chars}. Lower cfg.min_text_chars.")

if cfg.sample_random:
    work_df = valid.sample(n=cfg.n_rows, random_state=cfg.random_seed)
else:
    work_df = valid.head(cfg.n_rows)

work_df = work_df.reset_index(drop=False).rename(columns={"index": "_source_row"})
print("Selected rows:", len(work_df))
work_df[["_source_row", citation_col, "_text_len", text_col]].head(10)


In [ ]:
# Compact, question-neutral schema.
# IMPORTANT: no natural_language_queries, no query_phrases_en, no summary_en.

ENRICHMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "legal_area": {"type": "string"},
        "legal_domain_path": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 6
        },
        "topic": {"type": "string"},
        "subtopic": {"type": "string"},
        "micro_topic": {"type": "string"},
        "concepts_en": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 12
        },
        "terms_original": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 16
        },
        "statute_anchors": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "case_anchors": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "doctrinal_rule": {"type": "string"},
        "legal_test": {"type": "string"},
        "fact_pattern_tags": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "procedural_context": {"type": "string"},
        "paragraph_role": {
            "type": "string",
            "enum": ["holding", "reasoning", "facts", "procedural_history", "citation", "dissent", "neutral"]
        },
        "authority_role": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": [
                    "leading_decision",
                    "legal_test",
                    "constitutional_standard",
                    "statutory_interpretation",
                    "standard_of_review",
                    "application_of_rule",
                    "distinguishing_case",
                    "background",
                    "procedural_rule",
                    "evidentiary_standard",
                    "official_intervention_limit",
                    "voting_rights_jurisprudence",
                    "neutral"
                ]
            },
            "maxItems": 5
        },
        "outcome_signal": {
            "type": "string",
            "enum": ["granted", "dismissed", "remanded", "partially_granted", "inadmissible", "neutral"]
        },
        "specificity_score": {
            "type": "number",
            "minimum": 0,
            "maximum": 1
        }
    },
    "required": [
        "legal_area",
        "legal_domain_path",
        "topic",
        "subtopic",
        "micro_topic",
        "concepts_en",
        "terms_original",
        "statute_anchors",
        "case_anchors",
        "doctrinal_rule",
        "legal_test",
        "fact_pattern_tags",
        "procedural_context",
        "paragraph_role",
        "authority_role",
        "outcome_signal",
        "specificity_score"
    ],
    "additionalProperties": False
}

SYSTEM_PROMPT = """You are a deterministic Swiss legal citation indexing engine.

Return exactly one valid JSON object and nothing else.

Task:
Create compact, query-neutral retrieval metadata for one Swiss court citation.

Hard rules:
- Do NOT generate user questions.
- Do NOT write a general summary.
- Do NOT invent article numbers, statutes, cases, facts, outcomes, or procedural posture.
- Prefer precise legal concepts over generic labels.
- If a field is unclear, use an empty string, empty array, "neutral", or "background" as appropriate.
- Use English for taxonomy and concepts.
- Preserve important German/French/Italian legal terms in terms_original.
- Include statute_anchors and case_anchors only when explicitly present in the citation text.
- Output must satisfy the provided JSON schema.
"""

def make_user_prompt(citation: str, text: str) -> str:
    text = str(text).strip()
    if len(text) > cfg.text_chars:
        text = text[:cfg.text_chars].rsplit(" ", 1)[0] + " ..."

    schema_text = json.dumps(ENRICHMENT_SCHEMA, ensure_ascii=False, indent=2)

    return f"""Citation:
{citation}

Citation text:
{text}

Return JSON matching this schema:
{schema_text}
"""


In [ ]:
def extract_json_object(s: str) -> Dict[str, Any]:
    """Parse a JSON object from model output. Raises if the object is incomplete."""
    if not isinstance(s, str):
        raise ValueError("Model output is not a string.")
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)

    # First try exact JSON.
    try:
        return json.loads(s)
    except Exception:
        pass

    # Then find a balanced top-level JSON object.
    start = s.find("{")
    if start < 0:
        raise ValueError(f"No JSON object start found: {s[:300]}")

    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return json.loads(s[start:i+1])

    raise ValueError(f"No balanced JSON object found. Output was probably truncated: {s[:500]}")

def as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        return [x.strip()] if x.strip() else []
    return [str(x).strip()] if str(x).strip() else []

ROLE_VALUES = {"holding", "reasoning", "facts", "procedural_history", "citation", "dissent", "neutral"}
OUTCOME_VALUES = {"granted", "dismissed", "remanded", "partially_granted", "inadmissible", "neutral"}
AUTHORITY_VALUES = set(ENRICHMENT_SCHEMA["properties"]["authority_role"]["items"]["enum"])

def normalize_enrichment(obj: Dict[str, Any]) -> Dict[str, Any]:
    """Lenient normalization for preview. The model is guided, but this prevents one bad enum from killing the row."""
    out = {}
    for k in ENRICHMENT_SCHEMA["required"]:
        out[k] = obj.get(k, "")

    for k in ["legal_domain_path", "concepts_en", "terms_original", "statute_anchors", "case_anchors", "fact_pattern_tags", "authority_role"]:
        out[k] = as_list(out.get(k))

    for k in ["legal_area", "topic", "subtopic", "micro_topic", "doctrinal_rule", "legal_test", "procedural_context"]:
        out[k] = str(out.get(k, "") or "").strip()

    if out["paragraph_role"] not in ROLE_VALUES:
        out["paragraph_role"] = "neutral"

    if out["outcome_signal"] not in OUTCOME_VALUES:
        out["outcome_signal"] = "neutral"

    out["authority_role"] = [v for v in out["authority_role"] if v in AUTHORITY_VALUES]
    if not out["authority_role"]:
        out["authority_role"] = ["neutral"]

    try:
        out["specificity_score"] = float(out.get("specificity_score", 0.0))
    except Exception:
        out["specificity_score"] = 0.0
    out["specificity_score"] = max(0.0, min(1.0, out["specificity_score"]))

    return out

def build_retrieval_views(enrich: Dict[str, Any], citation: str, text: str) -> Dict[str, str]:
    return {
        "semantic_concepts_en": " ".join([
            enrich.get("legal_area", ""),
            enrich.get("topic", ""),
            enrich.get("subtopic", ""),
            enrich.get("micro_topic", ""),
            " ".join(enrich.get("concepts_en", [])),
            " ".join(enrich.get("fact_pattern_tags", [])),
        ]).strip(),
        "topic_path": " > ".join(enrich.get("legal_domain_path", [])),
        "original_terms_view": " ".join(enrich.get("terms_original", [])),
        "statute_anchor_view": " ".join(enrich.get("statute_anchors", [])),
        "case_anchor_view": " ".join([citation] + enrich.get("case_anchors", [])),
        "legal_rule_view": " ".join([enrich.get("doctrinal_rule", ""), enrich.get("legal_test", "")]).strip(),
        "raw_context": str(text)[:cfg.text_chars],
    }


In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional

# ---------------------------------------------------------------------
# Robust generator loader
# ---------------------------------------------------------------------
# It tries vLLM first for speed, while avoiding FlashInfer. If vLLM still
# cannot initialize on Kaggle, it automatically falls back to
# Transformers + GPTQModel AWQ, which is slower but should run.

import os
import gc

os.environ.pop("VLLM_ATTENTION_BACKEND", None)  # this was unknown in your vLLM logs
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ENGINE_TYPE = None
llm = None
model = None
tokenizer = None
SamplingParams = None
StructuredOutputsParams = None
GuidedDecodingParams = None


# ---------------------------------------------------------------------
# Validate local Kaggle model path
# ---------------------------------------------------------------------
model_path = Path(cfg.model_name)
if not model_path.exists():
    kaggle_model = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1")
    if kaggle_model.exists():
        cfg.model_name = str(kaggle_model)
        model_path = kaggle_model
        print("[model] cfg.model_name replaced with local Kaggle path:", cfg.model_name)
    else:
        raise FileNotFoundError(
            f"Configured model path does not exist: {model_path}\n"
            "Attach the Kaggle Qwen3-8B-AWQ model dataset or set cfg.model_name to the correct local model path."
        )

print("[model] using local path:", cfg.model_name)


def clear_cuda():
    try:
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def try_disable_flashinfer_import():
    """Best-effort: make FlashInfer unavailable before importing vLLM internals."""
    if not getattr(cfg, "avoid_flashinfer", True):
        return

    # If FlashInfer remains installed, vLLM may still auto-select it.
    # We do not uninstall here because pip operations in later cells are fragile.
    # Instead, we patch common import/selector points and rely on fallback if vLLM ignores them.
    patched = []

    try:
        import builtins
        real_import = builtins.__import__

        def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
            if name == "flashinfer" or name.startswith("flashinfer."):
                raise ImportError("FlashInfer disabled in this notebook to avoid Kaggle libcuda JIT link failure.")
            return real_import(name, globals, locals, fromlist, level)

        builtins.__import__ = guarded_import
        patched.append("builtins.__import__ flashinfer guard")
    except Exception as exc:
        print("[vLLM] import guard skipped:", repr(exc))

    if patched:
        print("[vLLM] FlashInfer import guard active.")


def load_vllm_engine() -> bool:
    """Return True if vLLM loaded successfully."""
    global llm, tokenizer, SamplingParams, StructuredOutputsParams, GuidedDecodingParams, ENGINE_TYPE

    if cfg.force_transformers or not cfg.use_vllm_first:
        print("[vLLM] skipped by config.")
        return False

    clear_cuda()
    try_disable_flashinfer_import()

    try:
        from vllm import LLM as _LLM, SamplingParams as _SamplingParams
        SamplingParams = _SamplingParams

        try:
            from vllm.sampling_params import StructuredOutputsParams as _StructuredOutputsParams
            StructuredOutputsParams = _StructuredOutputsParams
        except Exception:
            StructuredOutputsParams = None

        try:
            from vllm.sampling_params import GuidedDecodingParams as _GuidedDecodingParams
            GuidedDecodingParams = _GuidedDecodingParams
        except Exception:
            GuidedDecodingParams = None

    except Exception as exc:
        print("[vLLM] import failed; falling back to Transformers:", repr(exc))
        return False

    def base_kwargs() -> Dict[str, Any]:
        kwargs = dict(
            model=cfg.model_name,
            tensor_parallel_size=int(cfg.tensor_parallel_size),
            gpu_memory_utilization=float(cfg.gpu_memory_utilization),
            max_model_len=int(cfg.max_model_len),
            max_num_seqs=int(cfg.max_num_seqs),
            enforce_eager=bool(cfg.enforce_eager),
            trust_remote_code=True,
            disable_custom_all_reduce=bool(cfg.disable_custom_all_reduce),
            disable_log_stats=True,
        )
        if getattr(cfg, "quantization", None):
            kwargs["quantization"] = cfg.quantization
        return kwargs

    attempts = []

    k = base_kwargs()
    if cfg.force_triton_attention:
        # Try new/public API forms first. Unsupported kwargs are removed in later attempts.
        try:
            from vllm.config import AttentionConfig
            try:
                k["attention_config"] = AttentionConfig(backend="TRITON_ATTN")
            except Exception:
                k["attention_config"] = AttentionConfig(backend="triton_attn")
            print("[vLLM] using AttentionConfig for Triton attention")
        except Exception as exc:
            print("[vLLM] AttentionConfig unavailable:", repr(exc))
        k["attention_backend"] = "TRITON_ATTN"
    attempts.append(("vLLM with explicit Triton attention", dict(k)))

    k2 = dict(k)
    k2.pop("attention_backend", None)
    attempts.append(("vLLM without direct attention_backend kwarg", k2))

    k3 = dict(k2)
    k3.pop("attention_config", None)
    attempts.append(("vLLM minimal safe kwargs", k3))

    # Last vLLM attempt: conservative memory and no structured bells.
    k4 = dict(k3)
    k4.update(
        tensor_parallel_size=1,
        gpu_memory_utilization=min(float(cfg.gpu_memory_utilization), 0.58),
        max_model_len=min(int(cfg.max_model_len), 2560),
        max_num_seqs=min(int(cfg.max_num_seqs), 16),
        enforce_eager=True,
        quantization="awq",
    )
    attempts.append(("vLLM extra-conservative fallback", k4))

    last_exc = None
    for name, kwargs in attempts:
        print(f"[vLLM] load attempt: {name}")
        shown = {kk: vv for kk, vv in kwargs.items() if kk != "model"}
        print("[vLLM] kwargs:", shown)
        try:
            llm = _LLM(**kwargs)
            tokenizer = llm.get_tokenizer()
            ENGINE_TYPE = "vllm"
            print("[vLLM] model loaded successfully.")
            return True
        except TypeError as exc:
            last_exc = exc
            print("[vLLM] unsupported kwarg path:", repr(exc))
            continue
        except Exception as exc:
            last_exc = exc
            msg = str(exc).lower()
            print("[vLLM] failed:", repr(exc))
            # This is the exact failure mode from your logs.
            if "flashinfer" in msg or "cannot find -lcuda" in msg or "ninja build failed" in msg:
                print("[vLLM] FlashInfer/libcuda failure detected; switching to Transformers fallback.")
                break
            if "free memory" in msg and "gpu memory utilization" in msg:
                print("[vLLM] VRAM reservation failure; switching to Transformers fallback.")
                break
            continue

    print("[vLLM] all attempts failed. Last error:", repr(last_exc))
    return False


def load_transformers_engine() -> None:
    """Load Qwen3-8B-AWQ through Transformers + GPTQModel."""
    global model, tokenizer, ENGINE_TYPE

    clear_cuda()
    print("[Transformers] loading tokenizer:", cfg.model_name)

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        trust_remote_code=True,
        local_files_only=True,
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("[Transformers] loading AWQ model:", cfg.model_name)
    kwargs = dict(
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        local_files_only=True,
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model_name,
            dtype=torch.float16,
            **kwargs,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model_name,
            torch_dtype=torch.float16,
            **kwargs,
        )

    model.eval()
    ENGINE_TYPE = "transformers"

    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        print(f"[Transformers] model loaded. VRAM allocated={used:.2f} GiB")
    else:
        print("[Transformers] model loaded on CPU/GPU auto device map.")


def generate_texts(prompts: List[str], sampling_params=None) -> List[str]:
    """Unified generation API for vLLM and Transformers fallback."""
    global ENGINE_TYPE, llm, model, tokenizer

    if ENGINE_TYPE == "vllm":
        outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
        return [out.outputs[0].text if out.outputs else "" for out in outputs]

    if ENGINE_TYPE == "transformers":
        import torch

        max_input_tokens = max(512, int(cfg.max_model_len) - int(cfg.max_new_tokens))
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_tokens,
        )

        # Send inputs to the first model device. device_map='auto' still accepts inputs on cuda:0.
        if torch.cuda.is_available():
            inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

        gen_kwargs = dict(
            max_new_tokens=int(cfg.max_new_tokens),
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=float(cfg.repetition_penalty),
        )

        with torch.inference_mode():
            out = model.generate(**inputs, **gen_kwargs)

        prompt_len = inputs["input_ids"].shape[1]
        texts = tokenizer.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        return [t.strip() for t in texts]

    raise RuntimeError("No generation engine is loaded.")


# Load engine.
if not load_vllm_engine():
    load_transformers_engine()

print("Active generation engine:", ENGINE_TYPE)
print("Tokenizer ready:", tokenizer.__class__.__name__)


In [ ]:
def build_sampling_params():
    # Transformers fallback does not use vLLM SamplingParams.
    if ENGINE_TYPE != "vllm" or SamplingParams is None:
        print("Using Transformers generation parameters.")
        return None

    base = dict(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=cfg.max_new_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )

    # Prefer structured outputs when available.
    if StructuredOutputsParams is not None:
        try:
            sp = SamplingParams(
                **base,
                structured_outputs=StructuredOutputsParams(json=ENRICHMENT_SCHEMA),
            )
            print("Using vLLM structured_outputs JSON schema.")
            return sp
        except Exception as exc:
            print("structured_outputs unavailable:", repr(exc))

    # Fallback to guided decoding for older vLLM.
    if GuidedDecodingParams is not None:
        try:
            sp = SamplingParams(
                **base,
                guided_decoding=GuidedDecodingParams(json=ENRICHMENT_SCHEMA),
            )
            print("Using vLLM guided_decoding JSON schema.")
            return sp
        except Exception as exc:
            print("guided_decoding unavailable:", repr(exc))

    print("Using prompt-only JSON control.")
    return SamplingParams(**base)


def build_messages(row: pd.Series) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(str(row[citation_col]), str(row[text_col]))},
    ]


def render_prompt(messages: List[Dict[str, str]]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


sampling_params = build_sampling_params()


In [ ]:
out_dir = Path(cfg.output_dir)
out_jsonl = out_dir / cfg.output_jsonl
out_csv = out_dir / cfg.output_preview_csv
fail_jsonl = out_dir / cfg.output_failures_jsonl

records = []
failures = []
t0 = time.time()

for start in tqdm(range(0, len(work_df), cfg.batch_size), desc="enrich", unit="batch"):
    batch_df = work_df.iloc[start:start + cfg.batch_size]

    batch_rows = []
    prompts = []
    for _, row in batch_df.iterrows():
        r = {
            "_source_row": int(row["_source_row"]),
            "citation": str(row[citation_col]),
            "text": str(row[text_col]),
        }
        batch_rows.append(r)
        prompts.append(render_prompt(build_messages(row)))

    try:
        raw_outputs = generate_texts(prompts, sampling_params)
    except Exception as exc:
        # Batch-level failure: write every row to failures and continue.
        for r in batch_rows:
            failures.append({
                "_source_row": r["_source_row"],
                "citation": r["citation"],
                "text": r["text"],
                "error": "batch_generation_failed:" + repr(exc),
                "raw_output": "",
            })
        continue

    for r, raw in zip(batch_rows, raw_outputs):
        try:
            obj = extract_json_object(raw)
            enrich = normalize_enrichment(obj)
            rec = {
                "_source_row": r["_source_row"],
                "citation": r["citation"],
                "text": r["text"],
                "rag_enrichment": enrich,
                "retrieval_views": build_retrieval_views(enrich, r["citation"], r["text"]),
                "enrichment_quality": {
                    "method": "llm_legal_descriptor_enrichment",
                    "engine": ENGINE_TYPE,
                    "question_generation_used": False,
                    "summary_generation_used": False,
                    "grounded_references_only": True,
                    "low_value_paragraph": False,
                },
            }
            records.append(rec)
        except Exception as exc:
            failures.append({
                "_source_row": r["_source_row"],
                "citation": r["citation"],
                "text": r["text"],
                "error": repr(exc),
                "raw_output": raw,
            })

elapsed = time.time() - t0
print(f"Generated {len(records)} OK, {len(failures)} failed in {elapsed:.1f}s")
print(f"Rows/s: {len(work_df)/max(elapsed, 1e-9):.3f}")
print("Engine:", ENGINE_TYPE)

with out_jsonl.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with fail_jsonl.open("w", encoding="utf-8") as f:
    for rec in failures:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

if failures:
    print("Failures written to:", fail_jsonl)

# Flatten for easy visual inspection.
flat = []
for rec in records:
    enr = rec["rag_enrichment"]
    views = rec["retrieval_views"]
    flat.append({
        "_source_row": rec["_source_row"],
        "citation": rec["citation"],
        "legal_area": enr["legal_area"],
        "topic": enr["topic"],
        "subtopic": enr["subtopic"],
        "micro_topic": enr["micro_topic"],
        "concepts_en": " | ".join(enr["concepts_en"]),
        "terms_original": " | ".join(enr["terms_original"]),
        "statute_anchors": " | ".join(enr["statute_anchors"]),
        "case_anchors": " | ".join(enr["case_anchors"]),
        "doctrinal_rule": enr["doctrinal_rule"],
        "legal_test": enr["legal_test"],
        "fact_pattern_tags": " | ".join(enr["fact_pattern_tags"]),
        "procedural_context": enr["procedural_context"],
        "paragraph_role": enr["paragraph_role"],
        "authority_role": " | ".join(enr["authority_role"]),
        "outcome_signal": enr["outcome_signal"],
        "specificity_score": enr["specificity_score"],
        "semantic_concepts_en": views["semantic_concepts_en"],
        "original_terms_view": views["original_terms_view"],
    })

preview_df = pd.DataFrame(flat)
preview_df.to_csv(out_csv, index=False)

print("JSONL:", out_jsonl)
print("Preview CSV:", out_csv)
preview_df


In [ ]:
if records:
    print(json.dumps(records[0], ensure_ascii=False, indent=2)[:5000])
else:
    print("No successful records. First failure:")
    if failures:
        print(json.dumps(failures[0], ensure_ascii=False, indent=2)[:5000])


## What to inspect

For each row, check whether the enrichment is specific enough to distinguish the citation from generic court citations.

Good signs:
- `micro_topic` is narrow.
- `concepts_en` contains legally meaningful English descriptors.
- `terms_original` preserves exact German/French/Italian legal terms.
- `doctrinal_rule` and `legal_test` are short and grounded.
- `fact_pattern_tags` are specific, not generic.
- `retrieval_views.semantic_concepts_en` looks like a compact search document.

Bad signs:
- Generic labels like only "appeal", "court", "law", "decision".
- Invented statutes/cases.
- Synthetic user questions.
- Long summary-style prose.
